# Import Libraries

In [ ]:
import os
from pyspark.sql import SparkSession
import urllib.request
import plotly.express as px
from ipywidgets import interact, widgets
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
import pandas as pd








# Download the relevant datasets

In [ ]:
# 1. Define your raw URL and local destination path
raw_url = "https://raw.githubusercontent.com/nadilnemitha3-crypto/PySpark-DashBoard_Data_Assignment/refs/heads/main/before%20scaling.csv"
local_file = "dataset.csv"

scaled_url = "https://raw.githubusercontent.com/nadilnemitha3-crypto/PySpark-DashBoard_Data_Assignment/refs/heads/main/after%20scaling.csv"
scaled_local_file = "scaled_dataset.csv"



# Download both files to the Colab Environment

In [ ]:
# 2. Download the file locally to the Colab environment
urllib.request.urlretrieve(raw_url, local_file)
urllib.request.urlretrieve(scaled_url, scaled_local_file)

# 3. Start PySpark session
spark = SparkSession.builder.appName("ColabDashboard").getOrCreate()

# 4. Read the local file into PySpark
df = spark.read.csv(local_file, header=True, inferSchema=True)
df_scaled = spark.read.csv(scaled_local_file, header=True, inferSchema=True)

# Verification
df.show(5)

+-------+-------+----+-------------------+----------------+-----------------+----------------+----------------+------------+-------------------+
|Country|economy|Year|current_account_bal| domestic_credit|       gdp_growth|       inflation|    money_growth|unemployment|  exchange_rate_YoY|
+-------+-------+----+-------------------+----------------+-----------------+----------------+----------------+------------+-------------------+
|Albania|    ALB|1990|  -5.83174096323551|37.3109373018231|-9.57564016993408|226.005421253526|51.8198670705792|      10.304|                0.0|
|Albania|    ALB|1991|  -15.2788523176905|37.3109373018231|-28.0021416538074|226.005421253526|51.8198670705792|      10.304|                0.0|
|Albania|    ALB|1992|  -7.77398715257512|37.3109373018231|-7.18711090964234|226.005421253526|51.8198670705792|      30.007|                0.0|
|Albania|    ALB|1993|   1.25704931694863|37.3109373018231| 9.55941167111243|85.0047512387157|51.8198670705792|      25.251|  36.0

# Histogram

In [ ]:


numeric_cols = [

    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

def plot_histogram(selected_column, n_bins):
    # Extract column using PySpark and convert to Pandas
    pandas_df = df.select(selected_column).dropna().toPandas()

    # Setup plot
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(
        pandas_df[selected_column],
        bins=n_bins,
        kde=True,
        color="#1f77b4",
        ax=ax
    )

    # Styling
    ax.set_title(f"Distribution of {selected_column.replace('_', ' ').title()}", fontsize=14, pad=15)
    ax.set_xlabel(selected_column.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

# Interactive controls
interact(
    plot_histogram,
    selected_column=widgets.Dropdown(options=numeric_cols, value='gdp_growth', description='Attribute:'),
    n_bins=widgets.IntSlider(min=10, max=100, step=5, value=30, description='Bins:')
);

interactive(children=(Dropdown(description='Attribute:', options=('gdp_growth', 'inflation', 'unemployment', '…

# Bar Chart

In [ ]:
numeric_cols = [
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth']
def plot_country_bar_chart(selected_attribute, aggregation, top_n):
    # 1. Aggregation using PySpark
    if aggregation == 'Mean (Average)':
        spark_agg = df.groupBy("Country").agg(F.avg(selected_attribute).alias("Value"))
    else:
        spark_agg = df.groupBy("Country").agg(F.sum(selected_attribute).alias("Value"))

    # 2. Sort and limit to Top N countries in PySpark
    spark_agg = spark_agg.orderBy(F.col("Value").desc()).limit(top_n)

    # 3. Convert summary result to Pandas for plotting
    pandas_df = spark_agg.toPandas()

    # 4. Plotting using Matplotlib and Seaborn
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(
        data=pandas_df,
        x="Value",
        y="Country",
        hue="Country",        # Assigned hue to remove warning
        palette="viridis"
    )

    # Remove the redundant legend generated by assigning hue
    if ax.legend_:
        ax.legend_.remove()

    # Annotate bars with exact values
    for p in ax.patches:
        width = p.get_width()
        if not pd.isna(width):
            ax.annotate(
                f'{width:.2f}',
                (width, p.get_y() + p.get_height() / 2.),
                ha='left', va='center',
                xytext=(5, 0),
                textcoords='offset points',
                fontsize=9
            )

    # Chart Styling
    plt.title(f"Top {top_n} Countries by {aggregation} {selected_attribute.replace('_', ' ').title()}", fontsize=14, pad=15)
    plt.xlabel(f"{aggregation} {selected_attribute.replace('_', ' ').title()}", fontsize=12)
    plt.ylabel("Country", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.5, axis='x')

    plt.tight_layout()
    plt.show()

# Interactive Controls
interact(
    plot_country_bar_chart,
    selected_attribute=widgets.Dropdown(
        options=numeric_cols,
        value='gdp_growth',
        description='Attribute:'
    ),
    aggregation=widgets.Dropdown(
        options=['Mean (Average)', 'Sum'],
        value='Mean (Average)',
        description='Metric:'
    ),
    top_n=widgets.IntSlider(
        min=5,
        max=30,
        step=5,
        value=15,
        description='Top N:'
    )
);

interactive(children=(Dropdown(description='Attribute:', options=('gdp_growth', 'inflation', 'unemployment', '…

# Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from ipywidgets import interact, widgets

numeric_cols = [
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

# Fetch available unique countries dynamically from PySpark
country_list = ['All Countries'] + sorted([row.Country for row in df.select("Country").distinct().collect() if row.Country])
min_year = df.agg(F.min("Year")).collect()[0][0]
max_year = df.agg(F.max("Year")).collect()[0][0]

def plot_interactive_heatmap(selected_country, start_year, end_year, show_annotations):
    # 1. Filter dataset in PySpark by year range and country
    spark_filtered = df.filter(
        (F.col("Year") >= start_year) &
        (F.col("Year") <= end_year)
    )

    if selected_country != 'All Countries':
        spark_filtered = spark_filtered.filter(F.col("Country") == selected_country)

    # 2. Extract numeric columns to Pandas
    pandas_df = spark_filtered.select(numeric_cols).dropna().toPandas()

    if len(pandas_df) < 3:
        print(f"Not enough data points ({len(pandas_df)}) to compute correlations for the selected criteria.")
        return

    # 3. Compute Pearson correlation matrix
    corr_matrix = pandas_df.corr()

    # 4. Plot Heatmap
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        corr_matrix,
        annot=show_annotations,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        cbar_kws={'label': 'Pearson Correlation Coefficient'}
    )

    # Clean label formatting for display
    clean_labels = [c.replace('_', ' ').title() for c in numeric_cols]
    plt.xticks(range(len(numeric_cols)), clean_labels, rotation=45, ha='right')
    plt.yticks(range(len(numeric_cols)), clean_labels, rotation=0)

    title_text = f"Correlation Heatmap: {selected_country} ({start_year} - {end_year})"
    plt.title(title_text, fontsize=14, pad=15)

    plt.tight_layout()
    plt.show()

# 5. Render Interactive Controls
interact(
    plot_interactive_heatmap,
    selected_country=widgets.Dropdown(
        options=country_list,
        value='All Countries',
        description='Country:'
    ),
    start_year=widgets.IntSlider(
        min=min_year,
        max=max_year,
        step=1,
        value=1990,
        description='Start Year:'
    ),
    end_year=widgets.IntSlider(
        min=min_year,
        max=max_year,
        step=1,
        value=max_year,
        description='End Year:'
    ),
    show_annotations=widgets.Checkbox(
        value=True,
        description='Show Numbers'
    )
);

interactive(children=(Dropdown(description='Country:', options=('All Countries', 'Albania', 'Algeria', 'Angola…

# Time Series Plot

Select multiple countries by holding control button when selecting countries

In [ ]:
numeric_cols = [
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

# Fetch unique countries list from PySpark DataFrame
country_list = sorted([row.Country for row in df.select("Country").distinct().collect() if row.Country])
min_year = df.agg(F.min("Year")).collect()[0][0]
max_year = df.agg(F.max("Year")).collect()[0][0]

def plot_interactive_timeseries(selected_attribute, view_mode, selected_countries, start_year, end_year):
    # 1. Filter PySpark DataFrame by Year Range
    spark_filtered = df.filter(
        (F.col("Year") >= start_year) &
        (F.col("Year") <= end_year)
    )

    plt.figure(figsize=(12, 6))

    if view_mode == 'Compare Selected Countries':
        if not selected_countries:
            print("Please select at least one country from the multi-select box.")
            return

        # Filter selected countries
        spark_filtered = spark_filtered.filter(F.col("Country").isin(list(selected_countries)))
        pandas_df = spark_filtered.select("Country", "Year", selected_attribute).dropna().orderBy("Year").toPandas()

        # Line plot for multiple countries
        ax = sns.lineplot(
            data=pandas_df,
            x="Year",
            y=selected_attribute,
            hue="Country",
            marker="o",
            linewidth=2.5
        )
        plt.title(f"Time Series Comparison for {selected_attribute.replace('_', ' ').title()}", fontsize=14, pad=15)

    else:
        # Global Aggregate View (Mean and Median across all countries)
        spark_agg = spark_filtered.groupBy("Year").agg(
            F.avg(selected_attribute).alias("Mean"),
            F.expr(f"percentile_approx({selected_attribute}, 0.5)").alias("Median")
        ).orderBy("Year")

        pandas_df = spark_agg.toPandas()

        plt.plot(pandas_df["Year"], pandas_df["Mean"], label="Global Mean", marker="o", linewidth=2.5, color="#1f77b4")
        plt.plot(pandas_df["Year"], pandas_df["Median"], label="Global Median", marker="s", linewidth=2.5, color="#ff7f0e", linestyle="--")
        plt.legend(title="Metric")
        plt.title(f"Global Overall Trend for {selected_attribute.replace('_', ' ').title()}", fontsize=14, pad=15)

    # Chart Styling
    plt.xlabel("Year", fontsize=12)
    plt.ylabel(selected_attribute.replace('_', ' ').title(), fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

# Interactive Controls Setup
interact(
    plot_interactive_timeseries,
    selected_attribute=widgets.Dropdown(
        options=numeric_cols,
        value='gdp_growth',
        description='Attribute:'
    ),
    view_mode=widgets.Dropdown(
        options=['Compare Selected Countries', 'Global Aggregate (Mean vs Median)'],
        value='Compare Selected Countries',
        description='Mode:'
    ),
    selected_countries=widgets.SelectMultiple(
        options=country_list,
        value=['Albania', 'Algeria', 'Argentina'],
        description='Countries:',
        rows=6
    ),
    start_year=widgets.IntSlider(
        min=min_year,
        max=max_year,
        step=1,
        value=1990,
        description='Start Year:'
    ),
    end_year=widgets.IntSlider(
        min=min_year,
        max=max_year,
        step=1,
        value=max_year,
        description='End Year:'
    )
);

interactive(children=(Dropdown(description='Attribute:', options=('gdp_growth', 'inflation', 'unemployment', '…

# Distribution Analysis

In [ ]:
numeric_cols = [
    'cmri_score',
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

def analyze_distribution(selected_attribute, n_bins, show_kde, trim_outliers):
    # 1. Fetch data from PySpark
    selected_spark_df = df_scaled.select(selected_attribute).dropna()
    pandas_df = selected_spark_df.toPandas()

    data = pandas_df[selected_attribute]

    # 2. Optional Outlier Trimming (99th percentile filter for visualization)
    if trim_outliers:
        p99 = data.quantile(0.99)
        p01 = data.quantile(0.01)
        data = data[(data >= p01) & (data <= p99)]

    # 3. Calculate Summary Statistics using Pandas/PySpark
    mean_val = data.mean()
    median_val = data.median()
    std_val = data.std()
    skew_val = data.skew()
    kurt_val = data.kurtosis()

    # 4. Plot Histogram & KDE
    fig, ax = plt.subplots(figsize=(11, 5))

    sns.histplot(
        data,
        bins=n_bins,
        kde=show_kde,
        color="#1f77b4",
        edgecolor="white",
        ax=ax
    )

    # Draw Mean and Median vertical reference lines
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    ax.axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.2f}')

    # Styling
    attr_name = selected_attribute.replace('_', ' ').title()
    ax.set_title(f"Distribution Analysis of {attr_name}", fontsize=14, pad=15)
    ax.set_xlabel(attr_name, fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    # 5. Output Summary Statistics Box
    print("=" * 55)
    print(f" STATISTICAL SUMMARY FOR: {attr_name.upper()}")
    print("=" * 55)
    print(f" • Count          : {len(data):,}")
    print(f" • Mean           : {mean_val:.4f}")
    print(f" • Median         : {median_val:.4f}")
    print(f" • Std Deviation  : {std_val:.4f}")
    print(f" • Skewness       : {skew_val:.4f} ({'Right-skewed' if skew_val > 0 else 'Left-skewed'})")
    print(f" • Kurtosis       : {kurt_val:.4f}")
    print("=" * 55)

# Interactive Controls
interact(
    analyze_distribution,
    selected_attribute=widgets.Dropdown(
        options=numeric_cols,
        value='gdp_growth',
        description='Attribute:'
    ),
    n_bins=widgets.IntSlider(
        min=10,
        max=100,
        step=5,
        value=30,
        description='Bins:'
    ),
    show_kde=widgets.Checkbox(
        value=True,
        description='Overlay KDE Curve'
    ),
    trim_outliers=widgets.Checkbox(
        value=False,
        description='Trim Top/Bottom 1% Outliers'
    )
);

interactive(children=(Dropdown(description='Attribute:', index=1, options=('cmri_score', 'gdp_growth', 'inflat…

#Box Plot for Scaled Dataset

In [ ]:
# List of numeric columns to select
numeric_cols = [
    'cmri_score',
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

# Fetch available unique countries dynamically from PySpark
all_countries = sorted([row.Country for row in df_scaled.select("Country").distinct().collect() if row.Country])

def plot_interactive_boxplot(selected_attribute, compare_mode, top_n_countries):
    # 1. PySpark Aggregation / Filtering based on user choice
    if compare_mode == 'Top Countries (by Mean)':
        # Get top N countries by average value
        top_country_names = [
            row.Country for row in df_scaled.groupBy("Country")
            .agg(F.avg(selected_attribute).alias("avg_val"))
            .orderBy(F.col("avg_val").desc())
            .limit(top_n_countries)
            .collect()
        ]
        spark_filtered = df_scaled.filter(F.col("Country").isin(top_country_names))
    else:
        # Show global aggregate vs top 10
        spark_filtered = df_scaled

    # 2. Extract selected columns to Pandas
    pandas_df = spark_filtered.select("Country", selected_attribute).dropna().toPandas()

    # 3. Create Boxplot
    plt.figure(figsize=(12, 6))

    ax = sns.boxplot(
        data=pandas_df,
        x="Country",
        y=selected_attribute,
        hue="Country",
        palette="Set2"
    )

    # Clean up redundant legend from hue warning fix
    if ax.legend_:
        ax.legend_.remove()

    # Styling
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Distribution & Outliers of {selected_attribute.replace('_', ' ').title()}", fontsize=14, pad=15)
    plt.xlabel("Country", fontsize=12)
    plt.ylabel(selected_attribute.replace('_', ' ').title(), fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.4, axis='y')

    plt.tight_layout()
    plt.show()

# 4. Interactive Widgets Setup
interact(
    plot_interactive_boxplot,
    selected_attribute=widgets.Dropdown(
        options=numeric_cols,
        value='gdp_growth',
        description='Attribute:'
    ),
    compare_mode=widgets.Dropdown(
        options=['Top Countries (by Mean)', 'All Filtered Data'],
        value='Top Countries (by Mean)',
        description='Mode:'
    ),
    top_n_countries=widgets.IntSlider(
        min=3,
        max=15,
        step=1,
        value=7,
        description='Countries:'
    )
);

interactive(children=(Dropdown(description='Attribute:', index=1, options=('cmri_score', 'gdp_growth', 'inflat…

# Scatter Plot for the scaled dataset



In [ ]:
numeric_cols = [
    'cmri_score',
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

# Get year range from PySpark
min_year = df_scaled.agg(F.min("Year")).collect()[0][0]
max_year = df_scaled.agg(F.max("Year")).collect()[0][0]

def plot_interactive_scatterplot(x_attribute, y_attribute, start_year, end_year, add_trendline):
    # Check if the same attribute is selected for both axes
    if x_attribute == y_attribute:
        print("Please select two different attributes for the X-Axis and Y-Axis.")
        return

    # 1. Filter dataset in PySpark by year range and non-null values
    spark_filtered = df_scaled.filter(
        (F.col("Year") >= start_year) &
        (F.col("Year") <= end_year)
    ).select("Country", "Year", x_attribute, y_attribute).dropna()

    # 2. Convert filtered subset to Pandas for plotting
    pandas_df = spark_filtered.toPandas()

    # 3. Create Scatter Plot
    plt.figure(figsize=(10, 6))

    if add_trendline:
        # Plot with linear regression trendline
        sns.regplot(
            data=pandas_df,
            x=x_attribute,
            y=y_attribute,
            scatter_kws={'alpha': 0.5, 'color': '#1f77b4'},
            line_kws={'color': 'red', 'linewidth': 2}
        )
    else:
        # Standard scatter plot
        sns.scatterplot(
            data=pandas_df,
            x=x_attribute,
            y=y_attribute,
            alpha=0.6,
            color='#1f77b4'
        )

    # Styling
    x_label = x_attribute.replace('_', ' ').title()
    y_label = y_attribute.replace('_', ' ').title()

    plt.title(f"{y_label} vs. {x_label} ({start_year} - {end_year})", fontsize=14, pad=15)
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

# 4. Render Interactive Controls
interact(
    plot_interactive_scatterplot,
    x_attribute=widgets.Dropdown(
        options=numeric_cols,
        value='unemployment',
        description='X-Axis:'
    ),
    y_attribute=widgets.Dropdown(
        options=numeric_cols,
        value='gdp_growth',
        description='Y-Axis:'
    ),
    start_year=widgets.IntSlider(
        min=min_year,
        max=max_year,
        step=1,
        value=2000,
        description='Start Year:'
    ),
    end_year=widgets.IntSlider(
        min=min_year,
        max=max_year,
        step=1,
        value=max_year,
        description='End Year:'
    ),
    add_trendline=widgets.Checkbox(
        value=True,
        description='Show Trendline'
    )
);

interactive(children=(Dropdown(description='X-Axis:', index=3, options=('cmri_score', 'gdp_growth', 'inflation…

# Histogram for the scaled dataset

In [ ]:


numeric_cols = [
    'cmri_score',
    'gdp_growth',
    'inflation',
    'unemployment',
    'current_account_bal',
    'domestic_credit',
    'exchange_rate_YoY',
    'money_growth'
]

def plot_histogram(selected_column, n_bins):
    # Extract column using PySpark and convert to Pandas
    pandas_df = df_scaled.select(selected_column).dropna().toPandas()

    # Setup plot
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(
        pandas_df[selected_column],
        bins=n_bins,
        kde=True,
        color="#1f77b4",
        ax=ax
    )

    # Styling
    ax.set_title(f"Distribution of {selected_column.replace('_', ' ').title()}", fontsize=14, pad=15)
    ax.set_xlabel(selected_column.replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

# Interactive controls
interact(
    plot_histogram,
    selected_column=widgets.Dropdown(options=numeric_cols, value='gdp_growth', description='Attribute:'),
    n_bins=widgets.IntSlider(min=10, max=100, step=5, value=30, description='Bins:')
);

interactive(children=(Dropdown(description='Attribute:', index=1, options=('cmri_score', 'gdp_growth', 'inflat…